<a href="https://colab.research.google.com/github/tharushidambarage/BioLens-AI-Facial-Analyzer/blob/notebooks/01_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print(torch.__version__)
print("GPU:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
GPU: True
Tesla T4


In [2]:
import json
import os

kaggle_credentials = {
    "username": "tharushidambarage",
    "key": "KGAT_eeb56e1cf8b59c3a0456f6208fbb2e6d"
}

os.makedirs("/root/.kaggle", exist_ok=True)

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_credentials, f)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle API configured successfully!")

Kaggle API configured successfully!


In [3]:
!kaggle datasets list -s celeba

ref                                                   title                                               size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------  -------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
jessicali9530/celeba-dataset                          CelebFaces Attributes (CelebA) Dataset        1427750792  2018-06-01 20:08:48.043000         242288       1871  0.7647059        
kushsheth/face-vae                                    CelebA Dataset                                1427750792  2024-04-25 16:50:56.483000           2928         20  0.88235295       
badasstechie/celebahq-resized-256x256                 CelebA-HQ resized (256x256)                    296807591  2021-04-30 14:24:53.643000          19810         88  0.75             
quadeer15sh/celeba-face-recognition-triplets          CelebA Face Recognition Tr

In [6]:
# Move to Colab working directory
%cd /content

# Download CelebA dataset from Kaggle
!kaggle datasets download -d jessicali9530/celeba-dataset

/content
Dataset URL: https://www.kaggle.com/datasets/jessicali9530/celeba-dataset
License(s): other
celeba-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [7]:
import os

print("ZIP exists:", os.path.exists("/content/celeba-dataset.zip"))
print("ZIP size (GB):", round(os.path.getsize("/content/celeba-dataset.zip")/1024**3,2))

ZIP exists: True
ZIP size (GB): 1.33


In [11]:
import os

extract_path = "/content/celeba/img_align_celeba"

if os.path.exists(extract_path):
    print("Images extracted so far:", len(os.listdir(extract_path)))
else:
    print("Extraction folder not created yet.")

Images extracted so far: 1


In [12]:
import shutil
import os

if os.path.exists("/content/celeba"):
    shutil.rmtree("/content/celeba")

print("Partial extraction removed.")

Partial extraction removed.


In [14]:
!unzip -j /content/celeba-dataset.zip "list_attr_celeba.csv" -d /content/celeba_metadata

Archive:  /content/celeba-dataset.zip
  inflating: /content/celeba_metadata/list_attr_celeba.csv  


In [15]:
import pandas as pd

attr = pd.read_csv("/content/celeba_metadata/list_attr_celeba.csv")
attr.head()

,image_id,5_o_Clock_Shadow,Arched_Eyebrows,Attractive,Bags_Under_Eyes,Bald,Bangs,Big_Lips,Big_Nose,Black_Hair,...,Sideburns,Smiling,Straight_Hair,Wavy_Hair,Wearing_Earrings,Wearing_Hat,Wearing_Lipstick,Wearing_Necklace,Wearing_Necktie,Young
0,000001.jpg,-1,1,1,-1,-1,-1,-1,-1,-1,...,-1,1,1,-1,1,-1,1,-1,-1,1
1,000002.jpg,-1,-1,-1,1,-1,-1,-1,1,-1,...,-1,1,-1,-1,-1,-1,-1,-1,-1,1
2,000003.jpg,-1,-1,-1,-1,-1,-1,1,-1,-1,...,-1,-1,-1,1,-1,-1,-1,-1,-1,1
3,000004.jpg,-1,-1,1,-1,-1,-1,-1,-1,-1,...,-1,-1,1,-1,1,-1,1,1,-1,1
4,000005.jpg,-1,1,1,-1,-1,-1,1,-1,-1,...,-1,-1,-1,-1,-1,-1,1,-1,-1,1


In [16]:
import pandas as pd

attr = pd.read_csv("/content/celeba_metadata/list_attr_celeba.csv")

# Keep only the attributes we need
subset = attr[[
    "image_id",
    "Black_Hair",
    "Brown_Hair",
    "Blond_Hair",
    "Eyeglasses"
]].copy()

# Convert -1 to 0
for col in subset.columns[1:]:
    subset[col] = subset[col].replace(-1, 0)

# Keep images with one of the three hair colours
subset = subset[
    (subset["Black_Hair"] == 1) |
    (subset["Brown_Hair"] == 1) |
    (subset["Blond_Hair"] == 1)
]

# Random sample
subset = subset.sample(n=5000, random_state=42)

print("Selected images:", len(subset))
subset.head()

Selected images: 5000


,image_id,Black_Hair,Brown_Hair,Blond_Hair,Eyeglasses
1648,001649.jpg,1,0,0,0
58808,058809.jpg,0,1,0,0
76033,076034.jpg,0,1,0,0
112060,112061.jpg,0,1,0,0
39206,039207.jpg,1,0,0,0


In [17]:
subset.to_csv("/content/celeba_metadata/biolens_subset.csv", index=False)

In [18]:
# Extracting only 5000 images

In [19]:
import zipfile
import os
from tqdm import tqdm

zip_path = "/content/celeba-dataset.zip"
output_dir = "/content/BioLensSubset"

os.makedirs(output_dir, exist_ok=True)

# Read selected filenames
filenames = set(subset["image_id"])

with zipfile.ZipFile(zip_path, "r") as z:
    members = [m for m in z.namelist() if m.split("/")[-1] in filenames]

    for member in tqdm(members):
        z.extract(member, output_dir)

print("Extraction complete.")

100%|██████████| 5000/5000 [00:00<00:00, 7141.07it/s]

Extraction complete.


In [20]:
import glob

images = glob.glob("/content/BioLensSubset/**/*.jpg", recursive=True)

print("Images extracted:", len(images))
print(images[:5])

Images extracted: 5000
['/content/BioLensSubset/img_align_celeba/img_align_celeba/136145.jpg', '/content/BioLensSubset/img_align_celeba/img_align_celeba/080495.jpg', '/content/BioLensSubset/img_align_celeba/img_align_celeba/071991.jpg', '/content/BioLensSubset/img_align_celeba/img_align_celeba/000507.jpg', '/content/BioLensSubset/img_align_celeba/img_align_celeba/140268.jpg']


In [22]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
import os

print(os.path.exists("/content/drive/MyDrive"))
print(os.listdir("/content/drive"))

True
['MyDrive', '.shortcut-targets-by-id', '.Trash-0', '.Encrypted']


In [24]:
import os

base = "/content/drive/MyDrive/BioLens"

folders = [
    base,
    f"{base}/datasets",
    f"{base}/models",
    f"{base}/outputs",
    f"{base}/notebooks"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Folder structure created.")

✅ Folder structure created.


In [25]:
os.listdir("/content/drive/MyDrive/BioLens")

['datasets', 'models', 'outputs', 'notebooks']

In [26]:
!cp -r /content/BioLensSubset /content/drive/MyDrive/BioLens/datasets/
!cp /content/celeba_metadata/biolens_subset.csv /content/drive/MyDrive/BioLens/datasets/